In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import MultipleLocator, FixedLocator
import seaborn as sns
import plotly.express as px
from scipy.stats import norm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import itertools


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
from sklearn.impute import SimpleImputer

import sys
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk

import json

import time

from flax import nnx
import jax.numpy as jnp
import jax
import math 
import sys
import numpy as np
import optax
import netket as nk
import netket.experimental as nkx

In [ ]:
def regime(tFe):
    if tFe == 0:
        return 'Ferromagnético'
    else:
        return 'Antiferromagnético'    

In [ ]:
class Jastrow(nnx.Module):
    def __init__(self, N: int, *, rngs: nnx.Rngs):
        k1, k2 = jax.random.split(rngs.params())
        self.J = nnx.Param(0.01 * jax.random.normal(k1, (N, N),
                                                    dtype=jnp.complex128))

        self.v_bias = nnx.Param(0.01 * jax.random.normal(k2, (N, 1),
                                                         dtype=jnp.complex128))

    def __call__(self, x):
        x = x.astype(jnp.complex128)              # keep the dtypes aligned
        quad = jnp.einsum('...i,ij,...j->...', x, self.J, x)
        lin  = jnp.squeeze(x @ self.v_bias, -1)   # (...,N) @ (N,1) → (...,1)
        return quad + lin

In [ ]:
class Model(nnx.Module):

    def __init__(self, N: int, *, rngs: nnx.Rngs):
        self.linear = nnx.Linear(in_features=N, out_features=2 * N, dtype=jnp.complex128, rngs=rngs)

    def __call__(self, x: jax.Array):
        x = self.linear(x)
        x = nk.nn.activation.log_cosh(x)
        return jnp.sum(x, axis=-1)


class Model2(nnx.Module):

    def __init__(self, N: int, *, rngs: nnx.Rngs):
        self.linear1 = nnx.Linear(in_features=N, out_features=2 * N, dtype=jnp.complex128, rngs=rngs)
        self.linear2 = nnx.Linear(in_features=2 * N, out_features=N, dtype=jnp.complex128, rngs=rngs)

    def __call__(self, x: jax.Array):
        x = self.linear1(x)
        x = nk.nn.activation.log_cosh(x)
        x = self.linear2(x)
        x = nk.nn.activation.log_cosh(x)
        return jnp.sum(x, axis=-1)

In [ ]:
trained_params_list = []; parameters_list = [];iii = [] 
def conf(J,L):
    # Define custom graph
    edge_colors = []
    for i in range(L):
        edge_colors.append([i, (i + 1) % L, 1])
        edge_colors.append([i, (i + 2) % L, 2])

    # Define the netket graph object
    g = nk.graph.Graph(edges=edge_colors)

    sigmaz = [[1, 0], [0, -1]]
    mszsz = np.kron(sigmaz, sigmaz)

    # Exchange interactions
    exchange = np.asarray([[0, 0, 0, 0], [0, 0, 2, 0], [0, 2, 0, 0], [0, 0, 0, 0]])

    bond_operator = [
        (J[0] * mszsz).tolist(),
        (J[1] * mszsz).tolist(),
        (-J[0] * exchange).tolist(),
        (J[1] * exchange).tolist(),
    ]

    bond_color = [1, 2, 1, 2]
    hi = nk.hilbert.Spin(s=0.5, total_sz=0.0, N=g.n_nodes)
    op = nk.operator.GraphOperator(
        hi, graph=g, bond_ops=bond_operator, bond_ops_colors=bond_color
    )

    return g,hi,op

In [ ]:
### T = 0, ferromagnético; T=1; antiferromagnético
def jastrow_calc(T,L,it,v_pbc):
    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=v_pbc)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The Jastrow ground-state energy is ferromag E0='
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The Jastrow ground-state energy is antiferromag E0='

    ma = Jastrow(N=hi.size, rngs=nnx.Rngs(0))

    if T == 0:
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    else:
        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)
        sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=True)   
    
    op = nk.optimizer.Sgd(learning_rate=0.01)
    
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    j_out = 'dataf/jastrow_' + str(L) + '_' + str(T) + '_' + str(it)
    gs.run(it, out=j_out)
    end   = time.time()
    final_energy = float(gs.energy.mean.real)
    return final_energy,j_out

In [ ]:
def exac_calc_lanczos_ed(T,L,vpbc):
    g = nk.graph.Hypercube(length=L, n_dim=1, pbc=vpbc)
    if T == 0 :
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = -1.0 * nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens = 'The exact ground-state energy is ferromag E0='
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens = 'The exact ground-state energy is anti-ferromag E0='
        
    evals = nk.exact.lanczos_ed(ha, compute_eigenvectors=False)
    exact_gs_energy = evals[0]

    exact_df = pd.DataFrame()
    exact_v  = []
    exact_v.append(float(exact_gs_energy))
    exact_row_df = pd.DataFrame([exact_v])
    exact_df = pd.concat([exact_row_df])
    exact_df.insert(0, 'id', range(1, 1 + len(exact_df)))
    exact_df.columns = ['id','value']
    e_path = "dataf/exact_" + str(L)  + "_"  + str(T) + ".csv"

    exact_df.to_csv(e_path)

    return exact_gs_energy , e_path

In [ ]:
def info(e):
    head   = list(e.keys())[0]
    body   = list(e[head].keys())
    bias   = e[head][body[0]]
    kernel = e[head][body[1]]
    return  head, body, list(bias), list(kernel)
def real(c):
    return float(np.real(c))  
def img(c):
    return float(np.imag(c))    
def r_i(c):
    return real(c),img(c)  

def save_params(step, params, energy):
    trained_params_list.append(params.copy())
    parameters_list.append(energy.state.parameters.copy())
    iii.append(1)
    return True

In [ ]:
def run_net(T,L,it,vp,va,net):
    print(net) 
    if net == 'ffnn':
        return ffnn(T,L,it,vp,va)   
    elif net == 'rbm':
        return rbm(T,L,it,vp,va)
    elif net == 'ffnn_i':
        return ffnn_i(T,L,it,vp,va) 
    elif net == 'nffnn':
        return ffnn2(T,L,it,vp,va)         
    else:
        return  False

In [ ]:
def rbm(T,L,it,vp,va):
    paths = []

    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is ferromag E0='
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        ma = nk.models.RBM(alpha=va) 
        
        
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is antiferromag E0='

        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)
        ma = nk.models.RBM(alpha=va)

    
    
    op = nk.optimizer.Sgd(learning_rate=0.01)
    sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)

    opt = nk.optimizer.Sgd(learning_rate=0.01)

    sr = nk.optimizer.SR(diag_shift=0.01)
       
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    r_out = 'dataf/rbm_' + str(L) + '_' + str(T) + '_' + str(it) + '_' + str(va)
    gs.run(out=r_out, 
           n_iter=it,
           save_params_every=1,
          callback=save_params)
    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/rbm_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/rbm_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    final_energy = float(gs.energy.mean.real)

    return final_energy,r_out,paths 
    

ma = Model(N=hi.size, rngs=nnx.Rngs(1))
schedule = optax.linear_schedule(init_value=1e-3, end_value=1e-4, transition_steps=it)
op = nk.optimizer.Sgd(learning_rate=schedule)

vs = nk.vqs.MCState(sa, ma,
                   n_samples=16384,
                   n_chains=32,
                   n_discard_per_chain=200)

gs = nk.VMC(hamiltonian=ha, optimizer=op, preconditioner=sr, variational_state=vs)

In [ ]:
def ffnn_i(T, L, it, vp, va):
    paths = []

    g = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        #ha = (-2)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g, J=-1.0)

        sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)    

        sa = nk.sampler.MetropolisLocal(hilbert=hi, n_chains=32)  # mais cadeias
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        sa = nk.sampler.MetropolisExchange(hilbert=hi, graph=g)
        sr = nk.optimizer.SR(diag_shift=1e-2)

    # Modelo FFNN
    ma = Model(N=hi.size, rngs=nnx.Rngs(1))

    # Scheduler de learning rate
    schedule = optax.linear_schedule(
        init_value=1e-3, 
        end_value=1e-4,
        transition_steps=it
    )
    op = nk.optimizer.Sgd(learning_rate=schedule)

    # Mais amostras para FM
    vs = nk.vqs.MCState(sa, 
                        ma, 
                        n_samples=16384,
                        #n_chains=32,
                        n_discard_per_chain=200
                       )                   

    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs
    )

    r_out = f"dataf/ffnn_i_{L}_{T}_{it}"
    gs.run(out=r_out, n_iter=it, save_params_every=1, callback=save_params)

    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/ffnn_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/ffnn_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    final_energy = float(gs.energy.mean.real)
    return final_energy,r_out,paths 

In [ ]:
def ffnn(T, L, it, vp, va):
    paths = []
    g = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)

    if T == 0:
        # Configurações para regime ferromagnético
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        #ha = nk.operator.Heisenberg(hilbert=hi, graph=g, J=1.0)
        ha = (-1) * nk.operator.Heisenberg(hilbert=hi, graph=g, J=1.0)

        sa = nk.sampler.MetropolisLocal(hilbert=hi,  n_sweeps=10)
        sr = nk.optimizer.SR(diag_shift=0.05, holomorphic=False)
        op = nk.optimizer.RmsProp(learning_rate=0.0003  )
        n_samples = 4000  # Amostras aumentadas
        n_discard = 1000  # Thermalização mais longa

    else:
        # Configurações para regime antiferromagnético
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        sa = nk.sampler.MetropolisExchange(hilbert=hi, graph=g)
        sr = nk.optimizer.SR(diag_shift=0.1)  # Versão simplificada
        op = nk.optimizer.Adam(learning_rate=0.005)
        n_samples = 1008
        n_discard = 200


    ma = Model(N=hi.size, rngs=nnx.Rngs(1))
    op = nk.optimizer.Adam(learning_rate=0.001)  # Adam para melhor convergência
    vs = nk.vqs.MCState(sampler=sa, model=ma, n_samples=n_samples)
    
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs    )

    r_out = f"dataf/ffnn_{L}_{T}_{it}"
    gs.run(out=r_out, n_iter=it, save_params_every=1, callback=save_params)

    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/ffnn_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/ffnn_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    final_energy = float(gs.energy.mean.real)
    return final_energy,r_out,paths 


In [ ]:
def ffnn2(T,L,it,vp,va):
    paths = []

    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)


    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is ferromag E0='
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is antiferromag E0='
        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)

        
    
    ma = Model2(N=hi.size, rngs=nnx.Rngs(1))
    op = nk.optimizer.Sgd(learning_rate=0.01)
    sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)

    opt = nk.optimizer.Sgd(learning_rate=0.01)

    sr = nk.optimizer.SR(diag_shift=0.01)
       
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    r_out = 'dataf/ffnn2_' + str(L) + '_' + str(T) + '_' + str(it) 
    gs.run(out=r_out, 
           n_iter=it,
           save_params_every=1,
          callback=save_params)
    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/ffnn2_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/ffnn2_" + str(L)  + "_"  + str(T) + '_' + str(it)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")

    
    end = time.time()

    final_energy = float(gs.energy.mean.real)
    return final_energy,r_out,paths 

In [ ]:
def srbm(T,L,it,vp,va):

    paths = []

    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is ferromag E0='
        sa = nk.sampler.MetropolisLocal(hilbert=hi)
        ma = nk.models.RBM(alpha=va) 
        
        
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is antiferromag E0='

        sa = nk.sampler.MetropolisExchange(hilbert=hi,graph=g)
        ma = nk.models.RBMSymm(symmetries=g.translation_group(), alpha=va)
      
    
    
    op = nk.optimizer.Sgd(learning_rate=0.01)
    sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)

    opt = nk.optimizer.Sgd(learning_rate=0.01)

    sr = nk.optimizer.SR(diag_shift=0.01)
       
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    r_out = 'dataf/srbm_' + str(L) + '_' + str(T) + '_' + str(it) + '_' + str(va)
    gs.run(out=r_out, 
           n_iter=it,
           save_params_every=1,
          callback=save_params)
    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))

    path_nm = "dataf/srbm_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)
    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")



    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks.flatten():
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/srbm_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)
    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    final_energy = float(gs.energy.mean.real)
    return final_energy,r_out,paths 

In [ ]:
def rbm_fe(T,L,it,vp,va):

    paths = []
    g  = nk.graph.Hypercube(length=L, n_dim=1, pbc=vp)
    if T == 0:
        hi = nk.hilbert.Spin(s=0.5, N=g.n_nodes)
        ha = (-1)*nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is ferromag E0='
    else:
        hi = nk.hilbert.Spin(s=0.5, total_sz=0, N=g.n_nodes)
        ha = nk.operator.Heisenberg(hilbert=hi, graph=g)
        mens  = 'The RBM ground-state energy is antiferromag E0='
      

    ma = nk.models.RBM(alpha=va)
    sa = nk.sampler.MetropolisLocal(hilbert=hi)

    op = nk.optimizer.Sgd(learning_rate=0.01)
    sr = nk.optimizer.SR(diag_shift=0.1, holomorphic=False)
    vs = nk.vqs.MCState(sa, ma, n_samples=1008)
    gs = nk.VMC(
        hamiltonian=ha,
        optimizer=op,
        preconditioner=sr,
        variational_state=vs)
    start = time.time()
    
    r_out = 'dataf/rbm_fe_' + str(L) + '_' + str(T) + '_' + str(it) + '_' + str(va)

    gs.run(out=r_out, 
           n_iter=it,
           callback=save_params)


    ### <b> Results </b>
    head, body, bias_list,kernel_list = info(parameters_list[-1])

    real_df = pd.DataFrame()
    img_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for bias in bias_list:
            nr, ni = r_i(bias); 
            real_v.append(nr)
            img_v.append(ni)   
    
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
        
        real_df = pd.concat([real_df,real_row_df])
        img_df  = pd.concat([img_df,img_row_df])   


    real_df.insert(0, 'id', range(1, 1 + len(real_df)))
    img_df.insert(0, 'id', range(1, 1 + len(img_df)))


    path_nm = "dataf/rbm_fe_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)

    real_df.to_csv(path_nm + "_bias_real.csv",index=None) 
    paths.append(path_nm + "_bias_real.csv")
    img_df.to_csv(path_nm + "_bias_img.csv",index=None) 
    paths.append(path_nm + "_bias_img.csv")

    real_kernel_df = pd.DataFrame()
    img_kernel_df  = pd.DataFrame()
    for param in parameters_list:
        head, body, bias_list,kernel_list = info(param)
        real_v = [];img_v = []
        for ks in kernel_list:
            for k in ks:
                nr, ni = r_i(k); 
                real_v.append(nr)
                img_v.append(ni) 
        real_row_df = pd.DataFrame([real_v])
        img_row_df  = pd.DataFrame([img_v])
    
        real_kernel_df = pd.concat([real_kernel_df,real_row_df])
        img_kernel_df  = pd.concat([img_kernel_df,img_row_df])  

    real_kernel_df.insert(0, 'id', range(1, 1 + len(real_kernel_df)))
    img_kernel_df.insert(0, 'id', range(1, 1 + len(img_kernel_df)))

    path_nm = "dataf/rbm_fe_" + str(L) +  '_' + str(T)  + "_"  + str(it) + '_' + str(va)

    real_kernel_df.to_csv(path_nm + "_kernel_real.csv",index=None) 
    paths.append(path_nm + "_kernel_real.csv")
    img_kernel_df.to_csv(path_nm + "_kernel_img.csv",index=None) 
    paths.append(path_nm + "_kernel_img.csv")


    end = time.time()
    return gs.energy.mean.real,r_out, paths 